# Regression Models :: Full

In [ ]:
%load_ext autoreload
%autoreload 2

from datetime import date, timedelta

import warnings

warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = [10, 7]
plt.style.use("seaborn-v0_8")

import seaborn as sns

sns.set(style="darkgrid")

from lightgbm import LGBMRegressor
import numpy as np
import pandas as pd

from forecasting_sticker_sales.paths import PROJECT_DPATH
from forecasting_sticker_sales.features import create_features, get_holiday_df
from forecasting_sticker_sales.ts import get_date_string

## Constants

In [ ]:
SRC_DATA_DPATH = PROJECT_DPATH / "data" / "src_data"

DATA_DPATH = PROJECT_DPATH / "data" / "preprocessed_data"
assert DATA_DPATH.exists(), "Datasets are not initialized"

OUTPUT_DPATH = PROJECT_DPATH / "data" / "predictions"
OUTPUT_DPATH.mkdir(parents=True, exist_ok=True)

SPLIT_DATE = date(year=2017, month=1, day=1)
HORIZON = 1095  # 3 years

## Data Loading 

In [ ]:
train_fpath = DATA_DPATH / "train.csv"
train_df = pd.read_csv(train_fpath, parse_dates=["date"], index_col=0)
train_df.shape

In [ ]:
test_fpath = SRC_DATA_DPATH / "test.csv"
test_df = pd.read_csv(test_fpath, parse_dates=["date"])
test_df["num_sold"] = -1
test_df.shape

In [ ]:
df = pd.concat((train_df, test_df))
df.shape

In [ ]:
df

## Pipeline

In [ ]:
countries = df["country"].unique()
print(f"Unique Countries: {len(countries)}")

stores = df["store"].unique()
print(f"Unique Stores: {len(stores)}")

products = df["product"].unique()
print(f"Unique Products: {len(products)}")

print("\n-------------------\n")

iter_n = len(countries) * len(stores) * len(products)

missing_predictions = []
predictions = []

iter_counter = 1

for country in countries:
    for store in stores:
        for product in products:
            print(f"{iter_counter}/{iter_n} {country} - {store} - {product} processing ...")

            sample_df = df[
                (df["country"] == country) & (df["store"] == store) & (df["product"] == product)
            ]
            sample_df = sample_df[["num_sold", "date"]]
            sample_df = sample_df.resample("1D", on="date").sum()

            country_holiday_df = get_holiday_df(country)
            sample_df = create_features(sample_df, country_holiday_df, horizon=HORIZON)

            sample_train_df = sample_df.loc[: SPLIT_DATE - timedelta(days=1)]
            sample_test_df = sample_df.loc[SPLIT_DATE:]

            X_train = sample_train_df.drop(columns=["num_sold"])
            y_train = sample_train_df["num_sold"]
            X_test = sample_test_df.drop(columns=["num_sold"])

            iter_counter += 1

            # check data
            if X_train.empty or X_test.empty:
                missing_predictions.append((country, store, product))
                continue

            # forecasting
            model = LGBMRegressor(random_state=42, verbosity=-1)
            model.fit(X_train, y_train)
            y_pred = model.predict(X_test)

            # convert predictions to special format
            preds_df = test_df[
                (test_df["country"] == country)
                & (test_df["store"] == store)
                & (test_df["product"] == product)
            ]
            preds_df["num_sold"] = np.ceil(y_pred)
            predictions.append(preds_df)

predictions_df = pd.concat(predictions)
predictions_df.shape

In [ ]:
assert len(predictions_df) == len(test_df)

In [ ]:
missing_predictions

In [ ]:
predictions_df.head()

In [ ]:
predictions_df.tail()

In [ ]:
output_fpath = OUTPUT_DPATH / f"lgdm-prediction-{get_date_string(with_time=True)}.csv"
predictions_df[["id", "num_sold"]].to_csv(output_fpath, index=False)